### 1. Import Library
Mengimpor seluruh library yang dipakai di notebook ini: OpenCV & scikit-image untuk pengolahan citra, NumPy/Pandas untuk manipulasi data, dan scikit-learn untuk preprocessing, training, cross validation, serta evaluasi model SVM.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from skimage.feature import graycomatrix, graycoprops
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

### 2. Memuat Dataset Gambar
Fungsi `load_dataset` membaca dan me-resize seluruh gambar biji gandum di setiap folder kelas (`healthy_kernel`, `green_immature`, `shrunken_broken`, `insect_chewed`) menjadi ukuran 225x225 piksel.

In [ ]:
base_dir = 'dataset'

def load_dataset(folder, size=(225, 225)):
    images = []
    for files in os.listdir(folder):
        img_path = os.path.join(folder, files)
        img = cv2.imread(img_path)
        if img is not None:
            img_resized = cv2.resize(img, size)
            images.append((files, img_resized))
    return images


image_datasets = {
    'healthy_kernel': load_dataset(os.path.join(base_dir, 'healthy_kernel')),
    'green_immature': load_dataset(os.path.join(base_dir, 'green_immature')),
    'shrunken_broken': load_dataset(os.path.join(base_dir, 'shrunken_broken')),
    'insect_chewed': load_dataset(os.path.join(base_dir, 'insect_chewed'))
}

for label, images in image_datasets.items():
    print(f'{label}: {len(images)} images loaded.')

### 3. Ekstraksi Fitur Tekstur (GLCM)
Menghitung fitur tekstur Gray-Level Co-occurrence Matrix (contrast, dissimilarity, homogeneity, energy, correlation) dari citra grayscale.

In [ ]:
def extract_glcm_features(image):
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(gray_image, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)

    contrast = graycoprops(glcm, 'contrast')
    dissimilarity = graycoprops(glcm, 'dissimilarity')
    homogeneity = graycoprops(glcm, 'homogeneity')
    energy = graycoprops(glcm, 'energy')
    correlation = graycoprops(glcm, 'correlation')

    return [
        contrast[0, 0],
        dissimilarity[0, 0],
        homogeneity[0, 0],
        energy[0, 0],
        correlation[0, 0],
    ]

### 4. Ekstraksi Fitur Warna — Channel Green (RGB)
Mengambil rata-rata dan standar deviasi intensitas channel hijau (G) dari citra RGB.

In [ ]:
def extract_rgb_features(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    green_channel = image[:, :, 1]
    mean_green = np.mean(green_channel)
    std_green = np.std(green_channel)
    return [mean_green, std_green]

### 5. Ekstraksi Fitur Warna — Channel Value (HSV)
Mengambil rata-rata dan standar deviasi channel Value (V) dari citra HSV.

In [ ]:
def extract_hsv_features(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    v_channel = image[:, :, 2]
    mean_v = np.mean(v_channel)
    std_v = np.std(v_channel)
    return [mean_v, std_v]

### 6. Gabungan Seluruh Fitur
Menggabungkan fitur GLCM, RGB, dan HSV menjadi satu vektor fitur per gambar.

In [ ]:
def extract_all_features(image):
    glcm_features = extract_glcm_features(image)
    rgb_features = extract_rgb_features(image)
    hsv_features = extract_hsv_features(image)
    return glcm_features + rgb_features + hsv_features

print('Feature extraction functions ready!')

### 7. Ekstraksi Fitur untuk Seluruh Dataset
Menerapkan `extract_all_features` ke semua gambar di tiap kelas dan mengumpulkan hasilnya beserta labelnya masing-masing.

In [ ]:
features, labels = [], []

for label, images in image_datasets.items():
    print(f"Processing {label}...")
    for i, (filename, image_data) in enumerate(images):
        try:
            extracted_features = extract_all_features(image_data)
            features.append(extracted_features)
            labels.append(label)

            if (i + 1) % 100 == 0:
                print(f"  Processed {i + 1}/{len(images)} images")

        except Exception as e:
            print(f"  Error processing image {filename}: {e}")
            continue

print(f'\nExtracted features for {len(features)} images.')

### 8. Membentuk DataFrame
Menyusun fitur dan label yang telah diekstrak ke dalam `pandas.DataFrame` agar mudah dianalisis.

In [ ]:
glcm_columns = [
    'contrast_0',
    'dissimilarity_0',
    'homogeneity_0',
    'energy_0',
    'correlation_0',
]

rgb_columns = ['mean_g', 'std_g']

hsv_columns = ['mean_v', 'std_v']

columns = glcm_columns + rgb_columns + hsv_columns

df = pd.DataFrame(features, columns=columns)
df['Label'] = labels

print("DataFrame Info:")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst 5 rows:")
print(df.head())

### 9. Pemeriksaan Kualitas Data & Penyimpanan
Mengecek missing value dan tipe data, menampilkan statistik deskriptif, lalu menyimpan DataFrame ke file Excel dan CSV.

In [ ]:
print(f"NaN values per column:")
print(df.isnull().sum())

print(f"\nData types:")
print(df.dtypes)

print(f"\nBasic statistics for requested features:")
requested_features = ['mean_g', 'std_g', 'mean_v', 'std_v']
print(df[requested_features].describe())

df.to_excel(os.path.join('..', 'features_corrected.xlsx'), index=False)
df.to_csv(os.path.join('..', 'features_corrected.csv'), index=False)
print("\nFiles saved: features_corrected.xlsx and features_corrected.csv")

### 10. Distribusi Label
Menampilkan jumlah sampel per kelas untuk memastikan dataset seimbang (balanced).

In [ ]:
print("Label distribution:")
print(df['Label'].value_counts())

### 11. Split Data Training & Testing
Memisahkan fitur (`X`) dan label (`y`), lalu membagi data menjadi training set (80%) dan test set (20%) secara stratified.

In [ ]:
X = df.drop('Label', axis=1)
y = df['Label']

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Label distribution:")
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

### 12. Cross Validation (K=5)
Mengevaluasi robustness model menggunakan 5-fold Stratified Cross Validation pada data training saja (test set tidak disentuh), dengan scaling dilakukan di dalam pipeline agar tidak bocor antar-fold.

In [ ]:
pipeline_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC(random_state=42, max_iter=2000))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline_cv, X_train, y_train, cv=skf, scoring='accuracy')

print("5-Fold Cross Validation Accuracy per fold (data training):")
for i, score in enumerate(cv_scores, start=1):
    print(f"  Fold {i}: {score*100:.2f}%")

print(f"\nMean Accuracy : {cv_scores.mean()*100:.2f}%")
print(f"Std Deviation : {cv_scores.std()*100:.2f}%")

### 13. Scaling Fitur
Menstandardisasi fitur training dan test menggunakan `StandardScaler` yang di-fit hanya pada data training.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied (StandardScaler)")
print(f"Mean of scaled training features (should be ~0): {X_train_scaled.mean(axis=0).round(4)}")
print(f"Std of scaled training features (should be ~1): {X_train_scaled.std(axis=0).round(4)}")

### 14. Training Model SVM Final
Melatih model `LinearSVC` pada data training yang sudah di-scale.

In [ ]:
%%time
print("Training SVM Classifier... with LinearSVC() function")
print("="*50)

clf_LinearSVC = LinearSVC(random_state=42, max_iter=2000)
clf_LinearSVC.fit(X_train_scaled, y_train)

print("\u2713 SVM model trained successfully!")

### 15. Evaluasi Model pada Test Set
Mengukur performa model final pada test set yang belum pernah dilihat model, meliputi akurasi, classification report, dan confusion matrix.

In [ ]:
y_pred = clf_LinearSVC.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nDetailed Classification Report with LinearSVC:")
print("="*60)
print(classification_report(y_test, y_pred))
print("="*60)
fig, ax = plt.subplots(figsize=(6,6))
ConfusionMatrixDisplay.from_estimator(clf_LinearSVC, X_test_scaled, y_test, cmap="Blues", ax=ax)
plt.title("Confusion Matrix - Linear SVM")
plt.show()

### 16. Pengujian Skenario Kombinasi Fitur (5-Fold Cross Validation pada Data Training)
Membandingkan performa model pada 4 skenario kombinasi fitur berbeda (GLCM saja, warna saja, gabungan, dan gabungan tanpa normalisasi) menggunakan 5-fold Stratified Cross Validation **hanya pada `X_train`/`y_train`** — sama seperti cell CV model utama — agar `X_test` tetap murni tidak tersentuh sampai evaluasi akhir.

In [ ]:
skf_scenarios = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scenarios = {
    'S1 — GLCM saja': ['contrast_0', 'dissimilarity_0', 'homogeneity_0', 'energy_0', 'correlation_0'],
    'S2 — Warna saja': ['mean_g', 'std_g', 'mean_v', 'std_v'],
    'S3 — GLCM + Warna (gabungan)': ['contrast_0', 'dissimilarity_0', 'homogeneity_0', 'energy_0',
                                      'correlation_0', 'mean_g', 'std_g', 'mean_v', 'std_v'],
    'S4 — Gabungan tanpa normalisasi': ['contrast_0', 'dissimilarity_0', 'homogeneity_0', 'energy_0',
                                         'correlation_0', 'mean_g', 'std_g', 'mean_v', 'std_v'],
}

results = []

for name, feat_cols in scenarios.items():
    X_sc = X_train[feat_cols]
    y_sc = y_train

    if 'tanpa normalisasi' in name:
        model_sc = LinearSVC(random_state=42, max_iter=2000)
    else:
        model_sc = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', LinearSVC(random_state=42, max_iter=2000))
        ])

    cv_scores_sc = cross_val_score(model_sc, X_sc, y_sc, cv=skf_scenarios, scoring='accuracy')
    y_pred_cv = cross_val_predict(model_sc, X_sc, y_sc, cv=skf_scenarios)

    results.append({
        'Skenario': name,
        'Jumlah Fitur': len(feat_cols),
        'Mean Akurasi': f'{cv_scores_sc.mean()*100:.2f}%',
        'Std': f'{cv_scores_sc.std()*100:.2f}%',
    })

    print(f'\n{"="*55}')
    print(f'{name}')
    print(f'Akurasi per fold: {[f"{s*100:.2f}%" for s in cv_scores_sc]}')
    print(f'Mean Akurasi: {cv_scores_sc.mean()*100:.2f}% (± {cv_scores_sc.std()*100:.2f}%)')
    print(classification_report(y_sc, y_pred_cv))

print('\n' + '='*55)
print('RINGKASAN HASIL SKEMA PENGUJIAN (5-Fold CV pada Data Training)')
print('='*55)
for r in results:
    print(f"{r['Skenario']:<40} | {r['Jumlah Fitur']} fitur | {r['Mean Akurasi']} (± {r['Std']})")

Versi Library

In [ ]:
import cv2, skimage, numpy, pandas, sklearn, matplotlib
print("Python      :", __import__('sys').version)
print("OpenCV      :", cv2.__version__)
print("scikit-image:", skimage.__version__)
print("NumPy       :", numpy.__version__)
print("Pandas      :", pandas.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib  :", matplotlib.__version__)